# LangChain

## LangSmith 추적 설정

- LLM 앱 개발, 모니터링 및 테스트를 위한 플랫폼
- https://smith.langchain.com/ 접속하여 회원가입/로그인 진행

- 생성한 API Key를 넣고 설정

```.env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=API 키
LANGSMITH_PROJECT=프로젝트명
```

In [ ]:
%pip install python-dotenv

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
%pip install -U langchain langchain-openai

In [3]:
import os

def project_setup(name):
    os.environ['LANGSMITH_PROJECT'] = name
    os.environ['LANGSMITH_TRACING'] = 'true'
    print('프로젝트: {} 추적 시작'.format(name))

def stop_tracing():
    os.environ['LANGSMITH_TRACING'] = 'false'
    print('LangSmith 추적 중지')

project_setup('LangChain AI')

프로젝트: LangChain AI 추적 시작


In [4]:
MODEL = 'gpt-6-luna'

### LangChain 기초 

- Chat Model 호출

In [5]:
from langchain_openai import ChatOpenAI

# 객체 생성
llm = ChatOpenAI(
    temperature=0.1,  # 창의성 (0.0 ~ 2.0)
    reasoning={'effort': 'none'}, 
    max_tokens=2048, # 최대 토큰수
    model_name=MODEL,  # 모델명
)

In [6]:
response = llm.invoke("LangChain을 한 문장으로 설명해줘.")
print(type(response).__name__)   # AIMessage
print(response.content)

AIMessage
[{'type': 'text', 'text': 'LangChain은 대규모 언어 모델을 외부 데이터와 도구에 연결해 복잡한 AI 애플리케이션을 만들도록 돕는 프레임워크입니다.', 'annotations': [], 'id': 'msg_053674d1f6740506006adfed4d388887d0bff1cab76f23baa4', 'phase': 'final_answer'}]


In [7]:
print(response.content[0]['text'])

LangChain은 대규모 언어 모델을 외부 데이터와 도구에 연결해 복잡한 AI 애플리케이션을 만들도록 돕는 프레임워크입니다.


In [8]:
# AIMessage 구조 살펴보기
print("usage_metadata   :", response.usage_metadata)          # 입력 · 출력 · 전체 토큰
print("response_metadata:", response.response_metadata.get("model_name"),
      response.response_metadata.get("finish_reason"))
print("id               :", response.id)

usage_metadata   : {'input_tokens': 17, 'output_tokens': 44, 'total_tokens': 61, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}
response_metadata: gpt-6-luna None
id               : resp_053674d1f6740506006adfed4b428c87d0b9f44c4137c05971


In [12]:
# 질문
question = "대한민국의 수도는 어디인가요?"

# 답변
answer = llm.invoke(question)
print(f"[답변]: {answer}")

[답변]: content=[{'type': 'text', 'text': '대한민국의 수도는 서울특별시입니다.', 'annotations': [], 'id': 'msg_0774d38a6f428d32006adeafa04ff887d18713118aed54d404', 'phase': 'final_answer'}] additional_kwargs={} response_metadata={'id': 'resp_0774d38a6f428d32006adeaf9ed5c887d1957b296945728cdd', 'created_at': 1792978846.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'} id='resp_0774d38a6f428d32006adeaf9ed5c887d1957b296945728cdd' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 15, 'total_tokens': 30, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}


In [15]:
# Stream 방식
question = "대한민국의 아름다운 관광지 10곳과 주소를 알려주세요!"
answer = llm.stream(question)

for token in answer:
    if token.content:
        print(token.content[0]['text'], end="", flush=True)

대한민국의 아름다운 관광지 10곳과 주소를 소개해 드릴게요. 주소는 방문 전 최신 정보를 확인해 주세요.

| 관광지 | 주소 |
|---|---|
| 성산일출봉 | 제주특별자치도 서귀포시 성산읍 일출로 284-12 |
| 한라산국립공원 성판악탐방로 | 제주특별자치도 제주시 조천읍 516로 1865 |
| 우도 | 제주특별자치도 제주시 우도면 |
| 경복궁 | 서울특별시 종로구 사직로 161 |
| 남산서울타워 | 서울특별시 용산구 남산공원길 105 |
| 설악산국립공원 설악동탐방지원센터 | 강원특별자치도 속초시 설악산로 833 |
| 남이섬 | 강원특별자치도 춘천시 남산면 남이섬길 1 |
| 불국사 | 경상북도 경주시 불국로 385 |
| 해운대해수욕장 | 부산광역시 해운대구 해운대해변로 264 |
| 순천만국가정원 | 전라남도 순천시 국가정원1호길 47 |

한라산·설악산처럼 탐방로 운영이나 입산 시간이 달라질 수 있는 곳은 출발 전에 공식 홈페이지에서 예약·통제 여부를 확인하시면 좋습니다.

### 모델 추상화 - 같은 코드, 다른 모델

In [16]:
from langchain.chat_models import init_chat_model

luna = init_chat_model(MODEL, model_provider="openai")
sol = init_chat_model("gpt-6-sol", model_provider="openai")

question = [("human", "LLM의 할루시네이션을 한 문장으로 설명해줘.")]
for name, m in [("luna", luna), ("sol", sol)]:
    r = m.invoke(question)
    print(f"[{name}] {r.content}\n  → tokens: {r.usage_metadata}")

[luna] LLM의 할루시네이션은 모델이 사실이 아닌 내용을 그럴듯하고 자신 있게 생성하는 현상입니다.
  → tokens: {'input_tokens': 23, 'output_tokens': 30, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
[sol] LLM의 할루시네이션은 AI가 사실이 아닌 내용을 그럴듯하게 만들어내는 현상입니다.
  → tokens: {'input_tokens': 23, 'output_tokens': 29, 'total_tokens': 52, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [83]:
gemini = init_chat_model('google_genai:gemini-3.8-flash')

In [85]:
gemini.invoke(question)

AIMessage(content=[{'type': 'text', 'text': 'LLM의 할루시네이션(환각)이란 **인공지능이 사실이 아닌 거짓 정보를 마치 진실인 것처럼 그럴듯하게 지어내어 답변하는 현상**입니다.', 'extras': {'signature': 'EocQCoQQAWkUfRNu4ZfJw8F5fsBGXgH/HYxs5sPDAPjKN+noTF38Mu/XO//caxFKiwPVzvIseEd9XNXW9gX90XnQH9MX88Rwyj+O8Ryde+9XttIKSeLpLXQSmBEFoO8Qc1P0/wRPFzFYl8G64bE8KXKMirRHmNsKdSusceIZo2WUFOc1Ivx4vKVpauo4E7jygKKixQMale6be55Wm38OiqD6NQFCS5dr5T7SCpxjVfsXG9sYeYvSE84DLM3Y8NHVH0XixM1Y/FBXYTtMB38SOf/3KOimv5Bd4nQxHpcQkWTPIKo5+DcIeCoqwFDqtvyRdxLGwmJ1Bf2jdTUBSJz0+KAOP1p4BhWJ5lT37GjiRmo3XFXFW/4BzfUFsgsLq5zT/jmcH8eNtIVbtbDGvBmxSgwqN+neny3vxggfxq3SL2Uq/tXxPjjwtaSU/huaDL4jx7xxLhuBKiPRcRl7Wsmc+1zt6GtaYj7Tj4lL15N4S18kFegDGwGgh294nY/ZOhuut7lPdv9Qn/YL03HWXIdKEzhFxGua9uAmpC+81F2zi8ouWegVxcxouZShljwefNRVwfwCsmu6b2pDmhz5Eds+GydxNPFJ5hWeyp50pwS4S9B1Y4L4KvkmqO/H5cVGFjDsnBtv846l7aOp2xUfldf6ydQ1ey3U2B+2qmVGZaeTgfRbvW1cN9P58KmWsMGPoh1lexswZAxQsrrlf8D5W55pKADklbMdSoB3erjce7cWfBNYW/gtbtjGwm6EpY8JKn2nC0uKTl1LGA161t5c+P7PUk2JF4Jw8JBEgyGX0q+qYFsbILuUAbcgYhfSA5S7wBR1737SZ0DFyz9BvxaX7Xtd

In [ ]:
# 다른 공급자 사용 시 — 패키지만 설치하면 호출 코드는 동일

# %pip install langchain-anthropic      # ChatAnthropic
# ChatGoogleGenerativeAI
%pip install langchain-google-genai   
# %pip install langchain-ollama         # ChatOllama (로컬 모델)


## LCEL 기초

> #### 메시지 타입 & Role

> System Prompt : 사용자의 prompt 이전에 **성능 개선 용도**의 프롬프트로 *페르소나 부여*나 응답 어조 등 **중요 규칙**을 설정 (system)  
> User Prompt : 사용자의 입력  (user)  
> Assitant: AI 모델을 의미함 (ai)  
> Tool: 에이전트가 사용할 수 있는 도구 (tool)

In [28]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = [
    SystemMessage(content="당신은 친절한 파이썬 강사입니다. 2문장 이내로 답하세요."),
    HumanMessage(content="리스트와 튜플의 차이가 뭐예요?"),
]

[
    {'role': 'system', 'content': ''}
]

print(llm.invoke(messages).content)

[{'type': 'text', 'text': '리스트는 값을 추가·수정·삭제할 수 있는 **변경 가능한** 자료형이고, 대괄호 `[]`를 사용합니다. 튜플은 한 번 만들면 값을 바꿀 수 없는 **변경 불가능한** 자료형이며, 보통 소괄호 `()`를 사용합니다.', 'annotations': [], 'id': 'msg_088a35f719ead7ba006adeb4ec4a6487d180697a49211619d5', 'phase': 'final_answer'}]


In [29]:
# 이전 대화를 AIMessage로 넣어 맥락 제공
messages = [
    SystemMessage(content="당신은 친절한 파이썬 강사입니다."),
    HumanMessage(content="제 이름은 소신이에요."),
    AIMessage(content="반가워요, 소신님!"),
    HumanMessage(content="제 이름이 뭐라고 했죠?"),
]
print(llm.invoke(messages).content)

[{'type': 'text', 'text': '소신님이라고 하셨어요.', 'annotations': [], 'id': 'msg_0a11bcd47062c505006adeb50e8b9887d18443b6154791bda8', 'phase': 'final_answer'}]


In [30]:
# (role, content) 튜플 축약형
print(llm.invoke([("system", "한 단어로만 답해."), ("human", "대한민국의 수도는?")]).content)

[{'type': 'text', 'text': '서울', 'annotations': [], 'id': 'msg_050392b4e1e4cf07006adeb5267f9487d1babf7d87cf48f24c', 'phase': 'final_answer'}]


#### PromptTemplate

- 생성형 모델의 출력을 원하는 방향으로 유도하기 위한 입력 텍스트
- 문자열 템플릿

In [ ]:
from langchain_core.prompts import PromptTemplate

pt = PromptTemplate.from_template("{product}의 장점을 {n}가지 알려줘.")
print(pt.input_variables)
print(pt.format(product="무선 이어폰", n=3))
print(pt.invoke({"product": "무선 이어폰", "n": 3}))   # StringPromptValue

['n', 'product']
무선 이어폰의 장점을 3가지 알려줘.
text='무선 이어폰의 장점을 3가지 알려줘.'


In [35]:
chain = pt | llm
resp = chain.invoke({"product": "맥미니 M4", "n": 3})
resp

AIMessage(content=[{'type': 'text', 'text': '맥미니 M4의 장점은 다음 3가지입니다.\n\n1. **작지만 강력한 성능**: M4 칩으로 영상 편집, 사진 작업, 개발 등 다양한 작업을 빠르게 처리할 수 있습니다.\n2. **공간 절약과 간편한 설치**: 본체가 작아 책상 위 공간을 적게 차지하고, 모니터·키보드·마우스를 연결해 바로 사용할 수 있습니다.\n3. **다양한 연결성**: Thunderbolt, HDMI, USB-C 등 여러 포트를 갖춰 주변기기와 디스플레이를 연결하기 편리합니다.', 'annotations': [], 'id': 'msg_093a5c6470b097a9006adeb6d263e087d18b6144b01d8ba01d', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_093a5c6470b097a9006adeb6d1b8c487d18bb06ea37a886bb0', 'created_at': 1792980689.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_093a5c6470b097a9006adeb6d1b8c487d18bb06ea37a886bb0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 141, 'total_tokens': 161, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [36]:
print(resp.content[0]['text'])

맥미니 M4의 장점은 다음 3가지입니다.

1. **작지만 강력한 성능**: M4 칩으로 영상 편집, 사진 작업, 개발 등 다양한 작업을 빠르게 처리할 수 있습니다.
2. **공간 절약과 간편한 설치**: 본체가 작아 책상 위 공간을 적게 차지하고, 모니터·키보드·마우스를 연결해 바로 사용할 수 있습니다.
3. **다양한 연결성**: Thunderbolt, HDMI, USB-C 등 여러 포트를 갖춰 주변기기와 디스플레이를 연결하기 편리합니다.


#### ChatPromptTemplate

- 대화목록을 프롬프트로 주입하고자 할 때 활용
- 메시지 템플릿 

In [37]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        # role, message
        ("system", "당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name} 입니다."),
        ("human", "반가워요!"),
        ("ai", "안녕하세요! 무엇을 도와드릴까요?"),
        ("human", "{user_input}"),
    ]
)

In [38]:
messages = template.format_messages(
    name="제이슨", user_input="당신의 이름은 무엇입니까?"
)

messages

[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 제이슨 입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='반가워요!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='당신의 이름은 무엇입니까?', additional_kwargs={}, response_metadata={})]

In [39]:
llm.invoke(messages).content

[{'type': 'text',
  'text': '제 이름은 제이슨입니다. 👋',
  'annotations': [],
  'id': 'msg_0ff004e36a614eb8006adeb78e5e1887d19eb9eeaefa614ec1',
  'phase': 'final_answer'}]

In [40]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. {style} 말투로 답하세요."),
    ("human", "{question}"),
])

prompt_value = chat_prompt.invoke({
    "role": "여행 가이드", "style": "친근한", "question": "제주도 3월 여행 팁 알려줘",
})

for m in prompt_value.to_messages():
    print(type(m).__name__, ":", m.content)

print(llm.invoke(prompt_value).content)

SystemMessage : 당신은 여행 가이드입니다. 친근한 말투로 답하세요.
HumanMessage : 제주도 3월 여행 팁 알려줘
[{'type': 'text', 'text': '제주도 3월은 봄기운이 시작되지만 바람이 차고 날씨 변화가 잦아요. 준비만 잘하면 유채꽃과 한적한 풍경을 즐기기 좋은 시기예요!\n\n- **옷차림:** 얇은 옷을 여러 겹 입고, 바람막이나 가벼운 패딩을 챙기세요. 해안가와 오름은 체감온도가 더 낮을 수 있어요.\n- **날씨 확인:** 비와 강풍이 갑자기 올 수 있으니 출발 전 예보를 확인하고, 야외 일정은 실내 코스와 바꿀 수 있게 여유를 두세요.\n- **유채꽃 구경:** 산방산·섭지코지 주변 등에서 볼 수 있지만, 개화 시기는 해마다 달라요. 방문 직전에 현지 개화 상황을 확인하면 좋아요. 꽃밭은 사유지인 경우가 있으니 안내 표지와 입장 규칙을 지켜주세요.\n- **오름·한라산:** 바람이 강하면 정상에서 오래 머물기 어려워요. 미끄럽지 않은 운동화나 트레킹화를 신고, 한라산은 탐방로 통제와 예약 여부를 미리 확인하세요.\n- **이동:** 대중교통만으로는 시간이 더 걸릴 수 있어요. 렌터카를 이용한다면 미리 예약하고, 해안도로에서는 강풍과 갑작스러운 비에 특히 조심하세요.\n- **일정 짜기:** 하루에 동쪽·서쪽을 모두 돌기보다 한 지역씩 여유 있게 둘러보는 편이 좋아요. 비 오는 날을 대비해 박물관, 카페, 실내 관광지도 한두 곳 골라두세요.\n- **준비물:** 선크림, 보습제, 작은 우산이나 방수 재킷, 보조배터리를 챙기면 유용해요.\n\n3월 제주에서는 **바람을 고려해 옷을 입고, 꽃과 날씨 상황을 현지에서 확인하는 것**이 가장 중요해요.', 'annotations': [], 'id': 'msg_08108cc5dd88b5c5006adeb7b194a487d1980008ac96da6146', 'phase': 'final_answer'}]


In [41]:
# from_template: human 메시지 하나짜리 축약형 메서드
simple = ChatPromptTemplate.from_template("{topic}을(를) 초등학생에게 설명해줘.")
print(simple.invoke({"topic": "인공지능"}).to_messages())

[HumanMessage(content='인공지능을(를) 초등학생에게 설명해줘.', additional_kwargs={}, response_metadata={})]


#### partial 

- 템플릿의 일부 변수를 미리 고정하는 방법

In [42]:
from datetime import date

report_prompt = ChatPromptTemplate.from_messages([
    ("system", "오늘은 {today}입니다. 답변은 {lang}로 작성하세요."),
    ("human", "{question}"),
]).partial(today=str(date.today()), lang="한국어")

print(report_prompt.input_variables)   # ['question'] 만 남음
print(llm.invoke(report_prompt.invoke({"question": "오늘 날짜를 알려줘"})).content)

['question']
[{'type': 'text', 'text': '오늘은 2026년 10월 26일입니다.', 'annotations': [], 'id': 'msg_051180271e1c085c006adeb82637d487d1b958af3eac21716f', 'phase': 'final_answer'}]


In [43]:
report_prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={'today': '2026-10-26', 'lang': '한국어'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['lang', 'today'], input_types={}, partial_variables={}, template='오늘은 {today}입니다. 답변은 {lang}로 작성하세요.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

### 대화 · 예시 템플릿

#### MessagesPlaceholder — 대화 기록 자리

In [44]:
from langchain_core.prompts import MessagesPlaceholder

history_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 상담원입니다."),
    MessagesPlaceholder("history", optional=True),
    ("human", "{question}"),
])

history = [("human", "저는 채식주의자예요."), ("ai", "알겠습니다! 참고할게요.")]
pv = history_prompt.invoke({"history": history, "question": "점심 메뉴 하나 추천해줘"})
for m in pv.to_messages():
    print(type(m).__name__, ":", m.content)
print("\n→", llm.invoke(pv).content)

SystemMessage : 당신은 친절한 상담원입니다.
HumanMessage : 저는 채식주의자예요.
AIMessage : 알겠습니다! 참고할게요.
HumanMessage : 점심 메뉴 하나 추천해줘

→ [{'type': 'text', 'text': '채식주의자이시니 **두부 채소 비빔밥** 어떠세요? 고소한 두부와 신선한 채소를 듬뿍 넣고, 고추장은 따로 곁들이면 취향에 맞게 조절할 수 있어요.', 'annotations': [], 'id': 'msg_06a9f448e83c435e006adeb8b4092c87d1ac05978fa05f35df', 'phase': 'final_answer'}]


#### FewShotChatMessagePromptTemplate — 예시로 형식 맞추기

In [45]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "배송이 너무 늦어요", "output": "불만"},
    {"input": "영업시간이 어떻게 되나요?", "output": "문의"},
    {"input": "포장이 정말 예뻐요", "output": "칭찬"},
]
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
few_shot = FewShotChatMessagePromptTemplate(examples=examples, example_prompt=example_prompt)

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "고객 메시지를 [불만/문의/칭찬] 중 하나로만 답하세요."),
    few_shot,
    ("human", "{input}"),
])
print(llm.invoke(classify_prompt.invoke({"input": "환불은 언제 되나요?"})).content)

[{'type': 'text', 'text': '문의', 'annotations': [], 'id': 'msg_0d56a9d1107a48c7006adeb8fcfd7c87d1adbe10d2b32bac45', 'phase': 'final_answer'}]


In [ ]:
chain = classify_prompt | llm

chain.invoke({"input": "환불은 언제 되나요?"})

In [ ]:
f"""
고객 메시지를 [불만/문의/칭찬] 중 하나로만 답하세요.

예시)
"input": "배송이 너무 늦어요", "output": "불만"
"input": "영업시간이 어떻게 되나요?", "output": "문의"
"input": "포장이 정말 예뻐요", "output": "칭찬"

---
{question}
"""

In [48]:
for m in classify_prompt.invoke({"input": "환불은 언제 되나요?"}).to_messages():
    print(type(m).__name__, ":", m)

SystemMessage : content='고객 메시지를 [불만/문의/칭찬] 중 하나로만 답하세요.' additional_kwargs={} response_metadata={}
HumanMessage : content='배송이 너무 늦어요' additional_kwargs={} response_metadata={}
AIMessage : content='불만' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
HumanMessage : content='영업시간이 어떻게 되나요?' additional_kwargs={} response_metadata={}
AIMessage : content='문의' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
HumanMessage : content='포장이 정말 예뻐요' additional_kwargs={} response_metadata={}
AIMessage : content='칭찬' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
HumanMessage : content='환불은 언제 되나요?' additional_kwargs={} response_metadata={}


### Output Parsers (출력 파서)

- 출력을 원하는 구조화된 형태로 변환

#### StrOutputParser

-  AIMessage → 문자열

In [49]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
msg = llm.invoke("파이썬의 장점 하나만 한 문장으로")
print(type(msg).__name__, "→ str 여부:", isinstance(parser.invoke(msg), str))
print(parser.invoke(msg))

AIMessage → str 여부: True
파이썬은 문법이 간결해 배우고 사용하기 쉽습니다.


### JsonOutputParser

- 결과를 JSON 포맷으로 결과를 전달합니다.
- Pydantic 라이브러리를 활용하여 Json 스키마를 지정할 수 있습니다.

In [50]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_prompt = ChatPromptTemplate.from_template(
    "{topic}의 장점 3가지와 단점 3가지를 JSON으로만 답해.\n"
    '형식: {{"pros": ["", "", ""], "cons": ["", "", ""]}}'
)
# data = json_parser.invoke(llm.invoke(json_prompt.invoke({"topic": "재택근무"})))
# print(type(data), data)

In [54]:
import json

json.loads( llm.invoke(json_prompt.invoke({"topic": "재택근무"})).content[0]['text'] )

{'pros': ['출퇴근 시간이 줄어든다', '업무 시간과 장소를 유연하게 조정할 수 있다', '사무실 관련 비용을 절감할 수 있다'],
 'cons': ['동료와의 소통이 어려워질 수 있다',
  '업무와 개인 생활의 경계가 흐려질 수 있다',
  '집중하기 어려운 환경일 수 있다']}

In [55]:
# json_parser.invoke( llm.invoke(json_prompt.invoke({"topic": "재택근무"})) )

chain = json_prompt | llm | json_parser
resp = chain.invoke({"topic": "재택근무"})
resp

{'pros': ['출퇴근 시간이 줄어든다', '업무 환경을 원하는 대로 조성할 수 있다', '일과 개인 생활의 균형을 맞추기 쉽다'],
 'cons': ['동료와의 소통이 줄어들 수 있다', '업무와 휴식의 경계가 흐려질 수 있다', '집중하기 어려운 환경일 수 있다']}

In [56]:
type(resp)

dict

#### PydanticOutputParser

- `pydantic` 모듈의 `BaseModel`과 `Field`를 활용하여 객체의 구조를 정의

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [ ]:
from typing import Optional

class Movie(BaseModel):
    title: str = Field(description="영화 제목")
    year: int = Field(description="개봉 연도")
    genres: Optional[list[str]] = Field(description="장르 목록", strict=False)

movie_parser = PydanticOutputParser(pydantic_object=Movie)
print(movie_parser.get_format_instructions()[:300], "...")

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} ...


In [ ]:
print(movie_parser.get_format_instructions())

In [64]:
movie_prompt = ChatPromptTemplate.from_messages([
    ("system", "영화 정보를 알려주는 도우미입니다.\n{format_instructions}"),
    ("human", "{movie}에 대해 알려줘."),
]).partial(format_instructions=movie_parser.get_format_instructions())

In [66]:
movie = movie_parser.invoke(llm.invoke(movie_prompt.invoke({"movie": "기생충"})))
print(type(movie).__name__, movie.title, movie.year, movie.genres)

Movie 기생충 2019 ['드라마', '스릴러', '블랙 코미디']


In [70]:
type(movie.genres)

list

#### with_structured_output

- 모델의 구조화 출력 기능 사용 (권장)

In [ ]:
structured_llm = llm.with_structured_output(Movie)
movie2 = structured_llm.invoke("영화 '인터스텔라' 정보를 알려줘.")
print(type(movie2).__name__, movie2.model_dump())

Movie {'title': '인터스텔라', 'year': 2014, 'genres': ['SF', '드라마', '모험']}


In [73]:
movie2.title

'인터스텔라'

#### 기타 출력파서

> CommaSeparatedListOutputParser  
> StructuredOuputParser  
> PandasDataFrameOutputParser  
> DatetimeOutputParser  
> EnumOutputParser


### Runnable 인터페이스

- `Runnable` 클래스 (사용자 정의 체인을 쉽게 만들 수 있게 하는 프로토콜)

> `stream`: 응답의 청크를 스트리밍 (실시간 출력)  
> `invoke`: 입력에 대해 체인을 호출 (호출)  
> `batch`: 입력 목록에 대해 체인을 호출 (배치단위 호출)

- 비동기 방식도 존재
> `astream`, `ainvoke`, `abatch`, `astream_log`

### LangChain

- 여러 작업을 순차적으로 연결해서(Chain) 복잡한 워크플로우를 구축

In [74]:
chain = template | llm
# llm.invoke(template.invoke({}))

chain.invoke({"name": "Jason", "user_input": "당신의 이름은 무엇입니까?"}).content

[{'type': 'text',
  'text': '제 이름은 Jason입니다!',
  'annotations': [],
  'id': 'msg_0b456b4ad80fd258006adebee53b5087d1999ae482b1867890',
  'phase': 'final_answer'}]

In [77]:
model = ChatOpenAI(model=MODEL)

# 주어진 토픽에 대한 요약을 요청하는 프롬프트 템플릿을 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대하여 3문장으로 설명해줘.")

chain = prompt | model | StrOutputParser()

In [78]:
chain.invoke({
    'topic': 'ChatGPT'
})

'ChatGPT는 OpenAI가 개발한 인공지능 대화형 언어 모델입니다. 질문에 답하고 글쓰기, 요약, 번역, 아이디어 구상 등을 도와줍니다. 다만 때때로 부정확한 답변을 할 수 있으므로 중요한 정보는 확인하는 것이 좋습니다.'

In [79]:
# stream — 토큰 단위 스트리밍
for chunk in chain.stream({"topic": "양자 컴퓨터"}):
    print(chunk, end="", flush=True)
print()

양자 컴퓨터는 양자역학의 원리를 이용해 정보를 처리하는 컴퓨터입니다. 비트 대신 큐비트를 사용하며, 큐비트는 여러 상태의 가능성을 동시에 표현할 수 있습니다. 그래서 특정 문제는 기존 컴퓨터보다 훨씬 빠르게 풀 수 있지만, 모든 작업에서 더 빠른 것은 아닙니다.


In [82]:
# batch — 여러 입력을 병렬 처리
topics = [
    {"topic": "머신러닝"}, {"topic": "딥러닝"}, {"topic": "강화학습"}, 
    {'topic': 'ChatGPT'}, {'topic': 'Instagram'}, {'topic': '멀티모달'}]

results = chain.batch(topics, config={"max_concurrency": 3})
for t, r in zip(topics, results):
    print(f"[{t['topic']}] {r[:60]}...")

[머신러닝] 머신러닝은 컴퓨터가 데이터를 통해 패턴을 학습하고, 이를 바탕으로 예측하거나 판단하도록 하는 기술입니다. 예...
[딥러닝] 딥러닝은 여러 층으로 이루어진 인공신경망을 이용해 데이터에서 복잡한 패턴과 특징을 학습하는 기술입니다. 이미...
[강화학습] 강화학습은 에이전트가 환경과 상호작용하며 행동을 선택하고, 그 결과로 보상이나 벌점을 받는 학습 방법입니다....
[ChatGPT] ChatGPT는 OpenAI가 개발한 인공지능 대화 모델로, 사람의 질문을 이해하고 자연스러운 문장으로 답합...
[Instagram] 인스타그램은 사진과 동영상을 공유하고 소통하는 소셜 미디어 서비스입니다. 사용자는 게시물과 스토리, 릴스를 ...
[멀티모달] 멀티모달은 텍스트, 이미지, 음성, 영상처럼 서로 다른 형태의 정보를 함께 처리하는 방식입니다. 예를 들어 ...


In [ ]:
# 비동기 — ainvoke · abatch
# Jupyter에서는 셀에서 바로 `await main()` 사용 가능, .py로 실행할 때는 `asyncio.run(main())`

In [87]:
import asyncio

async def main():
    r1, r2 = await asyncio.gather(
        chain.ainvoke({"topic": "클라우드"}),
        chain.ainvoke({"topic": "엣지 컴퓨팅"}),
    )
    print(r1[:60], "...\n", r2[:60], "...")


try:
    loop = asyncio.get_running_loop()       # Jupyter는 이미 이벤트 루프가 실행 중
except RuntimeError:
    loop = None

if loop:
    print('주피터에서 실행')
    loop.create_task(main())                # Jupyter: `await main()` 으로 실행해도 동일
    # await main()
else:
    asyncio.run(main())                     # 일반 Python 스크립트 실행

주피터에서 실행


클라우드는 인터넷을 통해 서버, 저장 공간, 소프트웨어 같은 컴퓨터 자원을 이용하는 방식입니다. 내 기기에  ...
 엣지 컴퓨팅은 데이터를 중앙 클라우드가 아닌 데이터가 생성되는 기기나 가까운 네트워크에서 처리하는 기술입니다 ...


#### 입출력 스키마 확인

In [88]:
print("input :", chain.input_schema.model_json_schema())
print("output:", chain.output_schema.model_json_schema())

input : {'properties': {'topic': {'title': 'Topic', 'type': 'string'}}, 'required': ['topic'], 'title': 'PromptInput', 'type': 'object'}
output: {'title': 'StrOutputParserOutput', 'type': 'string'}


#### 기타 활용

In [ ]:
# JsonOutputParser 활용

In [90]:
# 원하는 데이터 구조를 정의합니다.
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [91]:
# 질의 작성
question = "지구 온난화의 심각성 대해 알려주세요."

# 파서를 설정하고 프롬프트 템플릿에 지시사항을 주입합니다.
parser = JsonOutputParser(pydantic_object=Topic)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser  # 체인을 구성합니다.

chain.invoke({"question": question})  # 체인을 호출하여 쿼리 실행

{'description': '지구 온난화는 온실가스 증가로 지구 평균기온이 상승하는 현상으로, 폭염·가뭄·홍수·산불과 해수면 상승을 심화시키고 생태계와 인간의 건강·식량·주거에 큰 피해를 줍니다. 온실가스 배출을 줄이고 기후변화에 대비하는 노력이 시급합니다.',
 'hashtags': '#지구온난화 #기후변화 #온실가스 #기후위기'}

In [93]:
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [97]:
# 질의 작성
question = "지구 온난화의 심각성 대해 알려주세요."

# 파서를 설정하고 프롬프트 템플릿에 지시사항을 주입합니다.
parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions='주제에 대한 간결한 설명: description, \
                        해시태그 형식의 키워드(2개 이상): hashtags로 구성된 json')

chain = prompt | llm | parser  # 체인을 구성합니다.

chain.invoke({"question": question})  # 체인을 호출하여 쿼리 실행

{'description': '지구 온난화는 온실가스 증가로 지구 평균기온이 상승하는 현상입니다. 폭염·가뭄·홍수·산불과 해수면 상승을 심화시키고 생태계와 인간의 건강, 식량·물 공급에 큰 위협을 줍니다. 피해를 줄이려면 온실가스 배출을 빠르게 감축하고 기후변화에 대비해야 합니다.',
 'hashtags': ['#지구온난화', '#기후위기', '#온실가스감축']}

In [95]:
# Pydantic 없이 사용가능

# 질의 작성
question = "지구 온난화에 대해 알려주세요. 온난화에 대한 설명은 `description`에, 관련 키워드는 `hashtags`에 담아주세요."

# JSON 출력 파서 초기화
parser = JsonOutputParser()

# 프롬프트 템플릿을 설정합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

# 지시사항을 프롬프트에 주입합니다.
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 프롬프트, 모델, 파서를 연결하는 체인 생성
chain = prompt | model | parser

# 체인을 호출하여 쿼리 실행
response = chain.invoke({"question": question})

# 출력을 확인합니다.
print(response)

{'description': '지구 온난화는 대기 중 온실가스가 늘어나면서 지구의 평균 기온이 장기적으로 상승하는 현상입니다. 화석연료 사용, 산림 파괴 등이 주요 원인이며, 폭염·가뭄·해수면 상승과 같은 기후 변화를 일으킵니다.', 'hashtags': ['#지구온난화', '#기후변화', '#온실가스', '#탄소배출', '#환경보호']}


In [96]:
parser.get_format_instructions()

'Return a JSON object.'

In [101]:
# PydanticOutputParser 활용
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [98]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")


# PydanticOutputParser 생성
parser = PydanticOutputParser(pydantic_object=EmailSummary)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [99]:

prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

# format 에 PydanticOutputParser의 부분 포맷팅(partial) 추가
prompt = prompt.partial(format=parser.get_format_instructions())

In [102]:
# chain 을 생성합니다.
chain = prompt | llm

# chain 을 실행하고 결과를 출력합니다.
output = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 JSON 형태로 출력됩니다.
print(output)

content=[{'type': 'text', 'text': '{\n  "person": "김철수",\n  "email": "chulsoo.kim@bikecorporation.me",\n  "subject": "\\"ZENESIS\\" 자전거 유통 협력 및 미팅 일정 제안",\n  "summary": "바이크코퍼레이션의 김철수 상무는 ZENESIS 자전거의 기술 사양, 배터리 성능, 디자인 정보가 담긴 상세 브로슈어를 요청했습니다. 이를 바탕으로 유통 전략과 마케팅 계획을 구체화하고, 유통 협력 가능성을 논의하고자 합니다.",\n  "date": "1월 15일 화요일 오전 10시"\n}', 'annotations': [], 'id': 'msg_0c485539f72b640c006aded3c3b3e487d1b122a6f32f48542b', 'phase': 'final_answer'}] additional_kwargs={} response_metadata={'id': 'resp_0c485539f72b640c006aded3c3285c87d1bd92bc1fe4f5fa0e', 'created_at': 1792988099.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'} id='resp_0c485539f72b640c006aded3c3285c87d1bd92bc1fe4f5fa0e' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 600, 'output_tokens': 148, 'total_tokens': 748, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_det

In [106]:
import json

data = json.loads(output.content[0]['text'])
data

{'person': '김철수',
 'email': 'chulsoo.kim@bikecorporation.me',
 'subject': '"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안',
 'summary': '바이크코퍼레이션의 김철수 상무는 ZENESIS 자전거의 기술 사양, 배터리 성능, 디자인 정보가 담긴 상세 브로슈어를 요청했습니다. 이를 바탕으로 유통 전략과 마케팅 계획을 구체화하고, 유통 협력 가능성을 논의하고자 합니다.',
 'date': '1월 15일 화요일 오전 10시'}

In [109]:
email_summary = EmailSummary(
    person=data['person'], 
    email=data['email'], 
    subject=data['subject'], 
    summary=data['summary'],
    date=data['date'])
email_summary

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션의 김철수 상무는 ZENESIS 자전거의 기술 사양, 배터리 성능, 디자인 정보가 담긴 상세 브로슈어를 요청했습니다. 이를 바탕으로 유통 전략과 마케팅 계획을 구체화하고, 유통 협력 가능성을 논의하고자 합니다.', date='1월 15일 화요일 오전 10시')

In [114]:
parser.invoke(output)

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션의 김철수 상무는 ZENESIS 자전거의 기술 사양, 배터리 성능, 디자인 정보가 담긴 상세 브로슈어를 요청했습니다. 이를 바탕으로 유통 전략과 마케팅 계획을 구체화하고, 유통 협력 가능성을 논의하고자 합니다.', date='1월 15일 화요일 오전 10시')

In [113]:
structured_output = parser.parse(output.content[0]['text'])
print(structured_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션의 김철수 상무는 ZENESIS 자전거의 기술 사양, 배터리 성능, 디자인 정보가 담긴 상세 브로슈어를 요청했습니다. 이를 바탕으로 유통 전략과 마케팅 계획을 구체화하고, 유통 협력 가능성을 논의하고자 합니다.' date='1월 15일 화요일 오전 10시'


In [115]:

# 출력 파서를 추가하여 전체 체인을 재구성합니다.
chain = prompt | llm | parser

# chain 을 실행하고 결과를 출력합니다.
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 EmailSummary 객체 형태로 출력됩니다.
response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션의 김철수 상무는 ZENESIS 자전거의 유통 협력을 논의하고자 기술 사양, 배터리 성능, 디자인 정보가 포함된 상세 브로슈어를 요청했습니다. 또한 유통 전략과 마케팅 계획을 구체화하기 위해 미팅을 제안했습니다.', date='1월 15일 화요일 오전 10시')